In [1]:
import pandas as pd
import sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'

%load_ext sql
%sql sqlite:///laliga.db

We read the CSV file into a pandas DataFrame.

In [2]:
df = pd.read_csv('Data/LaLiga_Matches.csv')

df.head()

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
0,1995-96,02-09-1995,La Coruna,Valencia,3,0,H,2.0,0.0,H
1,1995-96,02-09-1995,Sp Gijon,Albacete,3,0,H,3.0,0.0,H
2,1995-96,03-09-1995,Ath Bilbao,Santander,4,0,H,2.0,0.0,H
3,1995-96,03-09-1995,Ath Madrid,Sociedad,4,1,H,1.0,1.0,D
4,1995-96,03-09-1995,Celta,Compostela,0,1,A,0.0,0.0,D


We will now fix the missing data. These are not many, so we can just look the results up on the internet manually. 

The Valladolid - Betis game was actually won 2 - 1 by Valladolid, but oficially Betis won after suing Valladolid for illegal line-up [Link to the story](https://www.realbetisbalompie.es/noticias/la-historia/historia-del-2-1-al-0-3-21785). We will simply declare 0 - 0 at half-time.

In [3]:
mask = df.isna().any(axis=1)
df[mask]

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
136,1995-96,19-11-1995,Ath Bilbao,La Coruna,1,0,H,NaN,NaN,NaN
1472,1998-99,10-01-1999,Valladolid,Betis,0,3,A,NaN,NaN,NaN


In [4]:
df.loc[136,"HTHG"] = 0
df.loc[136,"HTAG"] = 0
df.loc[136,"HTR"] = 'D'

df.loc[1472,"HTHG"] = 0
df.loc[1472,"HTAG"] = 0
df.loc[1472,"HTR"] = 'D'

df["HTHG"] = df["HTHG"].astype(int)
df["HTAG"] = df["HTAG"].astype(int)

df[mask]

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
136,1995-96,19-11-1995,Ath Bilbao,La Coruna,1,0,H,0,0,D
1472,1998-99,10-01-1999,Valladolid,Betis,0,3,A,0,0,D


We also observed that the team Villarreal appears also under the name Villareal. We will keep only the former name.

In [5]:
df.replace("Villareal", "Villarreal", inplace=True)

An entry in the dataset consists of a game. The information given for each game is: season, date, home team, away team, full time home goals (FTHG), full time away goals (FTAG), full time result (FTR, Home H, Away A, Draw D), and similarly for half time HTHG, HTAG, HTR. 

We now start constructing our database. First, we will ensure that dates are stored in date format.

In [6]:
# We transform game dates into actual date format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Now we turn the DataFrame into a SQL table
df.to_sql('games', con='sqlite:///laliga.db', if_exists='replace', index=True, index_label='Game_id')

# And explore the table
%sql PRAGMA table_info(games);


 * sqlite:///laliga.db
Done.


cid,name,type,notnull,dflt_value,pk
0,Game_id,BIGINT,0,None,0
1,Season,TEXT,0,None,0
2,Date,DATETIME,0,None,0
3,HomeTeam,TEXT,0,None,0
4,AwayTeam,TEXT,0,None,0
5,FTHG,BIGINT,0,None,0
6,FTAG,BIGINT,0,None,0
7,FTR,TEXT,0,None,0
8,HTHG,BIGINT,0,None,0
9,HTAG,BIGINT,0,None,0


In [7]:
%sql SELECT * FROM games LIMIT 5;

 * sqlite:///laliga.db
Done.


Game_id,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
0,1995-96,1995-09-02 00:00:00.000000,La Coruna,Valencia,3,0,H,2,0,H
1,1995-96,1995-09-02 00:00:00.000000,Sp Gijon,Albacete,3,0,H,3,0,H
2,1995-96,1995-09-03 00:00:00.000000,Ath Bilbao,Santander,4,0,H,2,0,H
3,1995-96,1995-09-03 00:00:00.000000,Ath Madrid,Sociedad,4,1,H,1,1,D
4,1995-96,1995-09-03 00:00:00.000000,Celta,Compostela,0,1,A,0,0,D


In [8]:
# We will simplify the Season format from, e.g., "2019-2020" to just "19"
%sql UPDATE games\
SET Season = substr(Season, 3, 2);

 * sqlite:///laliga.db
11664 rows affected.


[]

In the next query, we create a table with season information.

In [9]:
%%sql 
DROP TABLE IF EXISTS seasons;

CREATE TABLE seasons (Season_id INTEGER PRIMARY KEY,
Season TEXT,
Number_of_Games INTEGER,
Number_of_Teams INTEGER,
Number_Of_Weeks INTEGER);

INSERT INTO seasons (Season, Number_of_Games, Number_of_Teams, Number_Of_Weeks)
SELECT Season, 
COUNT(*) AS Number_of_Games,
COUNT(DISTINCT HomeTeam) AS Number_of_Teams,
2*(COUNT(DISTINCT HomeTeam) - 1) AS Number_Of_Weeks 
FROM games
GROUP BY Season;

SELECT * FROM seasons;

 * sqlite:///laliga.db
Done.
Done.
31 rows affected.
Done.


Season_id,Season,Number_of_Games,Number_of_Teams,Number_Of_Weeks
1,00,380,20,38
2,01,380,20,38
3,02,380,20,38
4,03,380,20,38
5,04,380,20,38
6,05,380,20,38
7,06,380,20,38
8,07,380,20,38
9,08,380,20,38
10,09,380,20,38


We also add the season foreign key to the games table.

In [10]:
%%sql
ALTER TABLE games
ADD COLUMN Season_id INTEGER;

UPDATE games
SET Season_id =
(SELECT seasons.Season_id FROM seasons WHERE seasons.Season = games.Season);

SELECT * FROM games LIMIT 5;

 * sqlite:///laliga.db
Done.
11664 rows affected.
Done.


Game_id,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Season_id
0,95,1995-09-02 00:00:00.000000,La Coruna,Valencia,3,0,H,2,0,H,27
1,95,1995-09-02 00:00:00.000000,Sp Gijon,Albacete,3,0,H,3,0,H,27
2,95,1995-09-03 00:00:00.000000,Ath Bilbao,Santander,4,0,H,2,0,H,27
3,95,1995-09-03 00:00:00.000000,Ath Madrid,Sociedad,4,1,H,1,1,D,27
4,95,1995-09-03 00:00:00.000000,Celta,Compostela,0,1,A,0,0,D,27


Now we create a table with information about the teams, providing an id for each of them.

In [11]:
%%sql 
DROP TABLE IF EXISTS teams;

CREATE TABLE teams (
Team_id INTEGER PRIMARY KEY,
Team_Name TEXT UNIQUE,
Number_of_Seasons INTEGER,
Total_Games_Played INTEGER
);

INSERT INTO teams (Team_Name, Number_of_Seasons, Total_Games_Played)
SELECT HomeTeam AS Team,
COUNT(DISTINCT Season) AS Number_of_Seasons,
2*COUNT(*) AS Total_Games_Played
FROM Games
GROUP BY Team;

SELECT * FROM teams ORDER BY Number_of_Seasons DESC;

 * sqlite:///laliga.db
Done.
Done.
47 rows affected.
Done.


Team_id,Team_Name,Number_of_Seasons,Total_Games_Played
4,Ath Bilbao,31,1160
6,Barcelona,31,1156
34,Real Madrid,31,1158
42,Valencia,31,1158
5,Ath Madrid,29,1082
14,Espanol,29,1084
38,Sevilla,28,1044
39,Sociedad,28,1044
7,Betis,27,1006
45,Villarreal,26,960


Our next goal is to create standings tables. 

In [ ]:
games_sql = %sql SELECT * FROM games;
games_df = games_sql.DataFrame()

season_sql = %sql SELECT * FROM seasons;
season_df = season_sql.DataFrame()

fields = [
    "Game_id", "Next_Game_id", "Season_id", "Week", "Team",
    "Points",
    "Goals_For_Home_HT", "Goals_Against_Home_HT",
    "Goals_For_Away_HT", "Goals_Against_Away_HT",
    "Goals_For_Total_HT", "Goals_Against_Total_HT",
    "Goals_For_Home_FT", "Goals_Against_Home_FT",
    "Goals_For_Away_FT", "Goals_Against_Away_FT",
    "Goals_For_Total_FT", "Goals_Against_Total_FT",
    "Wins_Home", "Wins_Away", "Wins_Total",
    "Draws_Home", "Draws_Away", "Draws_Total",
    "Losses_Home", "Losses_Away", "Losses_Total",
    "Home_Games_Played", "Away_Games_Played", "Total_Games_Played",
    "Streak"
]

rows = []
for s in season_df["Season_id"]:
    teams_df = games_df.loc[games_df['Season_id']==s, ['HomeTeam']].drop_duplicates().rename(columns={'HomeTeam':'Name'})
    team_list = teams_df["Name"].tolist()
    number_of_weeks = season_df.loc[season_df['Season_id']==s, 'Number_Of_Weeks'].values[0]
    
    for team in team_list:
        history_df = games_df.loc[(games_df["Season_id"]==s) & ((games_df["HomeTeam"]==team) | (games_df["AwayTeam"]==team)), :]
        history_df = history_df.sort_values(by='Date',ascending=True).reset_index(drop=True)
        entry = pd.Series(0,index=fields,dtype=object)

        entry["Game_id"] = None
        entry["Next_Game_id"] = history_df["Game_id"].iat[0]
        entry["Season_id"] = s
        entry["Team"] = team
        entry["Streak"] = ""

        rows.append(entry.copy())

        for idx in range(len(history_df)):
            entry["Week"] = idx + 1
            if history_df['HomeTeam'].iat[idx] == team:
                entry['Goals_For_Home_HT'] += history_df["HTHG"].iat[idx]
                entry['Goals_Against_Home_HT'] += history_df["HTAG"].iat[idx]
                entry['Goals_For_Home_FT'] += history_df["FTHG"].iat[idx]
                entry['Goals_Against_Home_FT'] += history_df["FTAG"].iat[idx]
                if history_df['FTR'].iat[idx] == 'H':
                    entry['Wins_Home'] += 1
                    entry['Streak'] += 'W'
                elif history_df['FTR'].iat[idx] == 'D':
                    entry['Draws_Home'] += 1
                    entry['Streak'] += 'D'
                else:
                    entry['Losses_Home'] += 1
                    entry['Streak'] += 'L'
                entry['Home_Games_Played'] += 1
            elif history_df.iloc[idx]['AwayTeam'] == team:
                entry['Goals_For_Away_HT'] += history_df["HTAG"].iat[idx]
                entry['Goals_Against_Away_HT'] += history_df["HTHG"].iat[idx]
                entry['Goals_For_Away_FT'] += history_df["FTAG"].iat[idx]
                entry['Goals_Against_Away_FT'] += history_df["FTHG"].iat[idx]
                if history_df.iloc[idx]['FTR'] == 'A':
                    entry['Wins_Away'] += 1
                    entry['Streak'] += 'W'
                elif history_df.iloc[idx]['FTR'] == 'D':
                    entry['Draws_Away'] += 1
                    entry['Streak'] += 'D'
                else:
                    entry['Losses_Away'] += 1
                    entry['Streak'] += 'L'
                entry['Away_Games_Played'] += 1

            entry['Wins_Total'] = entry['Wins_Home'] + entry['Wins_Away']
            entry['Draws_Total'] = entry['Draws_Home'] + entry['Draws_Away']
            entry['Losses_Total'] = entry['Losses_Home'] + entry['Losses_Away']
            entry['Goals_For_Total_HT'] = entry['Goals_For_Home_HT'] + entry['Goals_For_Away_HT']
            entry['Goals_Against_Total_HT'] = entry['Goals_Against_Home_HT'] + entry['Goals_Against_Away_HT']
            entry['Goals_For_Total_FT'] = entry['Goals_For_Home_FT'] + entry['Goals_For_Away_FT']
            entry['Goals_Against_Total_FT'] = entry['Goals_Against_Home_FT'] + entry['Goals_Against_Away_FT']
            entry['Goals_For_Total'] = entry['Goals_For_Home_FT'] + entry['Goals_For_Away_FT']
            entry['Goals_Against_Total'] = entry['Goals_Against_Home_FT'] + entry['Goals_Against_Away_FT']
            entry['Total_Games_Played'] = entry['Home_Games_Played'] + entry['Away_Games_Played']
            entry['Points'] = (entry['Wins_Total'] * 3) + entry['Draws_Total']

            entry["Game_id"] = history_df["Game_id"].iat[idx]
            entry["Next_Game_id"] = history_df["Game_id"].iat[idx + 1] if idx + 1 < len(history_df) else None

            rows.append(entry.copy())

standings_df = pd.DataFrame(rows, columns=fields)
standings_df.head()
standings_df.to_sql('standings', con='sqlite:///laliga.db', if_exists='replace', index=True, index_label='standing_id')


 * sqlite:///laliga.db
Done.
 * sqlite:///laliga.db
Done.


C:\Users\JARDONH\AppData\Local\Temp\ipykernel_18552\281397062.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Barcelona' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  entry["Team"] = team
C:\Users\JARDONH\AppData\Local\Temp\ipykernel_18552\281397062.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'La Coruna' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  entry["Team"] = team
C:\Users\JARDONH\AppData\Local\Temp\ipykernel_18552\281397062.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Real Madrid' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  entry["Team"] = team
C:\Users\JARDONH\AppData\Local\Temp

23952

Let us now do a test query. We will retrieve the final classification for the season 1997-1998. This is the tragic season where Real Sporting de Gijon relegated after a still standing La Liga record: only 13 points obtained throughout the whole season. 

In [13]:
%%sql SELECT Team, Points
FROM standings 
WHERE Season_id = (SELECT Season_id FROM seasons WHERE Season = 97) 
AND Week = (SELECT Number_Of_Weeks FROM seasons WHERE Season = 97)
ORDER BY Points DESC;

 * sqlite:///laliga.db
Done.


Team,Points
Barcelona,74.0
Ath Bilbao,65.0
Real Madrid,63.0
Sociedad,63.0
Celta,60.0
Mallorca,60.0
Ath Madrid,60.0
Betis,59.0
Valencia,55.0
Espanol,53.0
